# Governance Table Setup

**Run once** to create all governance Delta tables with correct schemas and default configuration.

Requires: **Contributor** role + default Lakehouse attached.

### Tables created
| Table | Purpose |
|-------|---------|
| `cleanup_tracker` | Warning lifecycle state per item |
| `cleanup_audit_log` | Immutable log of all actions |
| `governance_config` | Configurable thresholds and rules |
| `email_outbox` | Generated emails queue for pipeline Outlook activity |
| `deleted_items_archive` | Full metadata snapshot of every item, written just before it's permanently deleted |


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from datetime import datetime, timezone

print("Creating governance Delta tables...")
print("=" * 50)


## 1. cleanup_tracker

In [ ]:
# NOTE: warning_count and cleanup_score are declared as IntegerType here, but every write
# path in this project (Tracker Update, Auto-Delete, Email Generator) writes the whole
# DataFrame via .astype(str) before saving, so in practice these columns are stored as
# strings — every consumer query in this project already accounts for this with
# CAST(... AS INT). Declared as StringType below to match what's actually on disk.
schema_tracker = StructType([
    StructField("item_id", StringType(), False),
    StructField("item_name", StringType(), True),
    StructField("item_type", StringType(), True),
    StructField("owner_email", StringType(), True),
    StructField("cleanup_score", StringType(), True),
    StructField("first_flagged_date", StringType(), True),
    StructField("warning_count", StringType(), True),
    StructField("warning_1_date", StringType(), True),
    StructField("warning_2_date", StringType(), True),
    StructField("warning_3_date", StringType(), True),
    StructField("status", StringType(), True),
    StructField("deleted_date", StringType(), True),
    StructField("resolved_date", StringType(), True),
    StructField("exemption_reason", StringType(), True),
    StructField("last_updated", StringType(), True),
    StructField("pipeline_run_id", StringType(), True),
    StructField("workspace_id", StringType(), True),
    # "true" once a deletion-confirmation email has been sent for this item — prevents
    # Governance_Email_Generator from re-notifying the owner on every future run.
    StructField("deletion_notified", StringType(), True),
])

df_empty = spark.createDataFrame([], schema_tracker)
df_empty.write.format("delta").mode("overwrite").saveAsTable("cleanup_tracker")
print("✔ cleanup_tracker created")


## 2. cleanup_audit_log

In [ ]:
schema_audit = StructType([
    StructField("audit_id", StringType(), False),
    StructField("timestamp", StringType(), True),
    StructField("pipeline_run_id", StringType(), True),
    StructField("item_id", StringType(), True),
    StructField("item_name", StringType(), True),
    StructField("item_type", StringType(), True),
    StructField("owner_email", StringType(), True),
    StructField("action", StringType(), True),
    StructField("detail", StringType(), True),
    StructField("workspace_id", StringType(), True),
])

df_empty = spark.createDataFrame([], schema_audit)
df_empty.write.format("delta").mode("overwrite").saveAsTable("cleanup_audit_log")
print("✔ cleanup_audit_log created")


## 3. email_outbox

In [ ]:
schema_outbox = StructType([
    StructField("email_id", StringType(), False),
    StructField("pipeline_run_id", StringType(), True),
    StructField("email_type", StringType(), True),
    StructField("recipient", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("body_html", StringType(), True),
    StructField("status", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("workspace_id", StringType(), True),
])

df_empty = spark.createDataFrame([], schema_outbox)
df_empty.write.format("delta").mode("overwrite").saveAsTable("email_outbox")
print("✔ email_outbox created")


## 3b. deleted_items_archive

In [ ]:
# Full metadata snapshot of every item at the moment it's permanently deleted.
# Written by Governance_Auto_Delete BEFORE the DELETE API call, so a wrongly-deleted
# item's complete governance record (owner, scores, dates, flags) is preserved even
# though the item itself is gone. This is what backs the "contact the administrator
# within 48 hours" recovery window promised in the deletion-confirmation email.
schema_archive = StructType([
    StructField("item_id", StringType(), False),
    StructField("item_name", StringType(), True),
    StructField("item_type", StringType(), True),
    StructField("workspace_id", StringType(), True),
    StructField("workspace_name", StringType(), True),
    StructField("owner_email", StringType(), True),
    StructField("description", StringType(), True),
    StructField("web_url", StringType(), True),
    StructField("created_date", StringType(), True),
    StructField("last_modified", StringType(), True),
    StructField("last_used_date", StringType(), True),
    StructField("cleanup_candidate_score", StringType(), True),
    StructField("is_stale", StringType(), True),
    StructField("is_unused_artifact", StringType(), True),
    StructField("has_missing_owner", StringType(), True),
    StructField("is_duplicate_name", StringType(), True),
    StructField("is_orphaned_model", StringType(), True),
    StructField("is_orphaned_endpoint", StringType(), True),
    StructField("warning_1_date", StringType(), True),
    StructField("warning_2_date", StringType(), True),
    StructField("warning_3_date", StringType(), True),
    StructField("deleted_by_pipeline_run_id", StringType(), True),
    StructField("archived_at", StringType(), True),
])

df_empty = spark.createDataFrame([], schema_archive)
df_empty.write.format("delta").mode("overwrite").saveAsTable("deleted_items_archive")
print("\u2714 deleted_items_archive created")

## 4. governance_config

In [ ]:
# Default configuration values
# NOTE: admin_email has a hardcoded fallback in Email_Generator/Auto_Delete/Failure_Notifier
# if this key is ever missing — those notebooks log a loud CRITICAL warning when that
# fallback kicks in rather than failing silently.
config_defaults = [
    ("cleanup_score_threshold", "30",    "Min score to enter cleanup workflow"),
    ("stale_cutoff_days",       "90",    "Days to flag as stale"),
    ("warnings_before_delete",  "3",     "Warnings before auto-delete"),
    ("days_between_warnings",   "1",     "Min days between warnings (1=testing, 7=production)"),
    ("admin_email",             "<ADMIN_EMAIL>", "Dashboard report recipient"),
    ("pipeline_schedule",       "daily_9am", "When pipeline runs"),
    ("protected_types",         "Lakehouse,Warehouse,Environment,SQLEndpoint", "Types never auto-deleted"),
    ("protected_items",         "",       "Comma-separated item IDs never deleted"),
    ("enable_auto_delete",      "false",  "Master switch for deletion (start disabled)"),
    ("workspace_ids",           "<WORKSPACE_ID>", "Comma-separated workspace IDs"),
    ("logo_url",                "",       "Xebia logo URL or base64 for emails"),
    ("dry_run",                 "true",   "Log actions without executing (true for testing)"),
    ("pipeline_failure_notify_email", "", "Recipient for pipeline activity-failure alerts (blank = falls back to admin_email)"),
]

schema_config = StructType([
    StructField("config_key", StringType(), False),
    StructField("config_value", StringType(), True),
    StructField("description", StringType(), True),
])

df_config = spark.createDataFrame(config_defaults, schema_config)
df_config.write.format("delta").mode("overwrite").saveAsTable("governance_config")
print("✔ governance_config created with defaults:")
display(spark.sql("SELECT config_key, config_value FROM governance_config ORDER BY config_key"))


In [ ]:
from PIL import Image
import io, base64

img = Image.open("/lakehouse/default/Files/Fabric_Monitoring/xebia_logo.png")

if img.mode != 'RGBA':
    img = img.convert('RGBA')

# Resize to 150px wide
ratio = 150 / img.size[0]
img_small = img.resize((150, int(img.size[1] * ratio)), Image.LANCZOS)

# White background version (logo is purple, needs light bg to be visible)
bg = Image.new('RGBA', img_small.size, (255, 255, 255, 255))
bg.paste(img_small, (0, 0), img_small)
img_final = bg.convert('RGB')

buf = io.BytesIO()
img_final.save(buf, format='PNG', optimize=True)
logo_data_uri = f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"

df_cfg = spark.sql("SELECT * FROM governance_config").toPandas()
df_cfg.loc[df_cfg["config_key"] == "logo_url", "config_value"] = logo_data_uri
spark.createDataFrame(df_cfg).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("governance_config")
print(f"✔ Real Xebia logo saved ({len(logo_data_uri)} chars)")

## 5. Verify all tables

In [ ]:
%%sql
-- Verify all tables exist and show row counts
SELECT 'cleanup_tracker' AS table_name, COUNT(*) AS row_count FROM cleanup_tracker
UNION ALL
SELECT 'cleanup_audit_log', COUNT(*) FROM cleanup_audit_log
UNION ALL
SELECT 'email_outbox', COUNT(*) FROM email_outbox
UNION ALL
SELECT 'governance_config', COUNT(*) FROM governance_config
UNION ALL
SELECT 'workspace_inventory_snapshot', COUNT(*) FROM workspace_inventory_snapshot
UNION ALL
SELECT 'deleted_items_archive', COUNT(*) FROM deleted_items_archive


## Done

All tables created. You can now run the **Governance_Tracker_Update** notebook.

To modify thresholds later:
```sql
UPDATE governance_config SET config_value = '50' WHERE config_key = 'cleanup_score_threshold'
```
